In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root:", project_root)
print("Python executable:", sys.executable)

Project root: c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project
Python executable: c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\.venv\Scripts\python.exe


In [3]:
import pandas as pd

entity_df = pd.read_csv("../data/processed/entity_catalog.csv")
relation_df = pd.read_csv("../data/processed/relation_catalog.csv")

print(f"Entities: {len(entity_df)}")
print(f"Relations: {len(relation_df)}")

Entities: 3265
Relations: 375


In [4]:
from src.kg.rdf_builder import build_rdf_graph

graph = build_rdf_graph(entity_df, relation_df)
print("Number of triples:", len(graph))

Number of triples: 2441


In [5]:
from pathlib import Path

Path("../kg_artifacts").mkdir(exist_ok=True)

graph.serialize("../kg_artifacts/initial_graph.ttl", format="turtle")
print("Sauvegardé → kg_artifacts/initial_graph.ttl")

Sauvegardé → kg_artifacts/initial_graph.ttl


In [13]:
from src.kg.wikidata_linker import link_entities

# test sur les équipes seulement
teams = entity_df[entity_df["entity_type"] == "Team"][["entity_text", "entity_type"]].drop_duplicates().to_dict("records")
print(f"Test sur {len(teams)} équipes...")

linked_teams = link_entities(teams)

for r in linked_teams:
    status = r["wikidata_qid"] or "NOT FOUND"
    print(f"{r['entity_text']:25} | {status:12} | {r['confidence']} | {r['wikidata_description']}")

Test sur 10 équipes...
Alpine                    | Q98930155    | 0.8 | French Formula One racing team
Aston Martin              | Q104620704   | 0.8 | Formula One racing team
Audi                      | Q120756696   | 0.8 | Formula One team
Cadillac                  | Q131323328   | 0.8 | Formula One team and constructor
Ferrari                   | Q2009683     | 0.8 | Ferrari F1-2000 Formula 1 car
Haas F1 Team              | Q16300222    | 1.0 | Formula 1 team
McLaren                   | Q172030      | 1.0 | British Formula One team
Mercedes                  | Q172721      | 0.8 | auto racing team
Red Bull Racing           | Q173663      | 1.0 | Formula One racing team based in Milton Keynes, England
Williams                  | Q171337      | 0.8 | British Formula One team and constructor


In [12]:
from src.kg.wikidata_linker import search_wikidata_entity

# test avec nom enrichi
for name in ["Alpine F1 Team", "Audi F1", "Cadillac F1", "Mercedes F1"]:
    result = search_wikidata_entity(name, "Team")
    if result:
        print(f"{name:25} | {result['qid']:12} | {result['confidence']} | {result['description']}")
    else:
        print(f"{name:25} | NOT FOUND")

Alpine F1 Team            | Q98930155    | 1.0 | French Formula One racing team
Audi F1                   | Q120756696   | 0.8 | Formula One team
Cadillac F1               | Q131323328   | 0.6 | Formula One team and constructor
Mercedes F1               | Q172721      | 0.8 | auto racing team


In [19]:
linked_teams = link_entities(teams)

for r in linked_teams:
    status = r["wikidata_qid"] or "NOT FOUND"
    print(f"{r['entity_text']:25} | {status:12} | {r['confidence']} | {r['wikidata_description']}")

Alpine                    | Q98930155    | 1.0 | French Formula One racing team
Aston Martin              | Q104620704   | 0.8 | Formula One racing team
Audi                      | Q120756696   | 1.0 | Formula One team
Cadillac                  | Q131323328   | 0.6 | Formula One team and constructor
Ferrari                   | Q169898      | 1.0 | Formula One team
Haas F1 Team              | Q16300222    | 1.0 | Formula 1 team
McLaren                   | Q172030      | 1.0 | British Formula One team
Mercedes                  | Q172721      | 1.0 | auto racing team
Red Bull Racing           | Q173663      | 1.0 | Formula One racing team based in Milton Keynes, England
Williams                  | Q171337      | 0.8 | British Formula One team and constructor


In [10]:
import requests

WIKIDATA_SEARCH_URL = "https://www.wikidata.org/w/api.php"
HEADERS = {"User-Agent": "F1-KG-Project/1.0 (student project; paula@example.com)"}

params = {
    "action": "wbsearchentities",
    "search": "Scuderia Ferrari",
    "language": "en",
    "format": "json",
    "limit": 5,
}
response = requests.get(WIKIDATA_SEARCH_URL, params=params, headers=HEADERS, timeout=10)
for r in response.json()["search"]:
    print(r["id"], "|", r["label"], "|", r.get("description", ""))

Q169898 | Scuderia Ferrari | Formula One team
Q110930131 | Scuderia Ferrari Race 2013 | 2013 video game
Q5866426 | history of Scuderia Ferrari | aspect of history
Q2274853 | Ferrari Grand Prix results | Wikimedia list article


In [14]:
from src.kg.wikidata_linker import search_wikidata_entity
result = search_wikidata_entity("Ferrari", "Team")
print(result)

{'uri': 'http://www.wikidata.org/entity/Q2009683', 'qid': 'Q2009683', 'label': 'Ferrari F1-2000', 'description': 'Ferrari F1-2000 Formula 1 car', 'confidence': 0.8}


In [18]:
for search in ["Cadillac Formula One", "Cadillac F1", "Mercedes AMG F1", "Mercedes F1 team"]:
    params = {
        "action": "wbsearchentities",
        "search": search,
        "language": "en",
        "format": "json",
        "limit": 3,
    }
    response = requests.get(WIKIDATA_SEARCH_URL, params=params, headers=HEADERS, timeout=10)
    results = response.json().get("search", [])
    print(f"\n{search}:")
    for r in results:
        print(f"  {r['id']} | {r['label']} | {r.get('description', '')}")


Cadillac Formula One:

Cadillac F1:
  Q131323328 | Cadillac Formula 1 Team | Formula One team and constructor

Mercedes AMG F1:
  Q173358 | Mercedes AMG High Performance Powertrains | British Formula One engine manufacturer
  Q28450077 | Mercedes AMG F1 W08 EQ Power+ | Mercedes AMG Petronas F1 Team 2017 Formula 1 car
  Q48965018 | Mercedes AMG F1 W09 EQ Power+ | Mercedes-AMG Petronas Motorsport 2018 Formula 1 car

Mercedes F1 team:
  Q172721 | Mercedes F1 Team | auto racing team


In [20]:
# test sur les drivers principaux seulement
key_drivers = [
    "Lewis Hamilton", "Max Verstappen", "Michael Schumacher",
    "Ayrton Senna", "Fernando Alonso", "Sebastian Vettel",
    "Charles Leclerc", "Lando Norris", "George Russell"
]

driver_rows = [{"entity_text": name, "entity_type": "Driver"} for name in key_drivers]
linked_drivers = link_entities(driver_rows)

for r in linked_drivers:
    status = r["wikidata_qid"] or "NOT FOUND"
    print(f"{r['entity_text']:25} | {status:12} | {r['confidence']} | {r['wikidata_description']}")

Lewis Hamilton            | Q9673        | 1.0 | British racing driver
Max Verstappen            | Q2239218     | 1.0 | Dutch and Belgian racing driver
Michael Schumacher        | Q9671        | 1.0 | German racing driver
Ayrton Senna              | Q10490       | 1.0 | Brazilian racing driver (1960-1994)
Fernando Alonso           | Q10514       | 1.0 | Spanish racing driver
Sebastian Vettel          | Q42311       | 1.0 | German racing driver
Charles Leclerc           | Q17541912    | 1.0 | Monegasque racing driver
Lando Norris              | Q22007193    | 1.0 | British and Belgian racing driver (born 1999)
George Russell            | Q17319645    | 1.0 | British racing driver (born 1998)


In [21]:
# on garde seulement les types qu'on sait bien gérer
types_to_link = ["Driver", "Team", "GrandPrix", "Season"]

entities_to_link = (
    entity_df[entity_df["entity_type"].isin(types_to_link)]
    [["entity_text", "entity_type"]]
    .drop_duplicates()
    .to_dict("records")
)

print(f"Entités à linker: {len(entities_to_link)}")

Entités à linker: 939


In [22]:
# on garde seulement les entités avec au moins 2 mots pour Driver
# et on exclut les entités qui contiennent des chiffres seuls
def is_worth_linking(row):
    text = row["entity_text"]
    entity_type = row["entity_type"]
    
    # exclut les textes trop courts
    if len(text) < 4:
        return False
    # exclut les textes qui commencent par un chiffre
    if text[0].isdigit():
        return False
    return True

entities_filtered = [r for r in entities_to_link if is_worth_linking(r)]
print(f"Après filtrage: {len(entities_filtered)} entités")

Après filtrage: 840 entités


In [23]:
from src.kg.wikidata_linker import link_entities

print("Linking en cours... (peut prendre 4-5 minutes)")
linked = link_entities(entities_filtered, delay=0.3)

# stats rapides
found = [r for r in linked if r["wikidata_uri"]]
not_found = [r for r in linked if not r["wikidata_uri"]]

print(f"\nRésultats:")
print(f"  Trouvés    : {len(found)}")
print(f"  Non trouvés: {len(not_found)}")

Linking en cours... (peut prendre 4-5 minutes)

Résultats:
  Trouvés    : 475
  Non trouvés: 365


In [24]:
import pandas as pd

linked_df = pd.DataFrame(linked)

# sauvegarde
linked_df.to_csv("../kg_artifacts/entity_alignment_table.csv", index=False)
print("Sauvegardé → kg_artifacts/entity_alignment_table.csv")

# inspecte les non trouvés par type
print("\nNon trouvés par type:")
not_found_df = linked_df[linked_df["wikidata_uri"].isna()]
print(not_found_df["entity_type"].value_counts())

print("\nExemples non trouvés (Driver):")
print(not_found_df[not_found_df["entity_type"] == "Driver"]["entity_text"].head(20).tolist())

Sauvegardé → kg_artifacts/entity_alignment_table.csv

Non trouvés par type:
entity_type
Driver       282
GrandPrix     83
Name: count, dtype: int64

Exemples non trouvés (Driver):
['- van de Burgt', 'Agencia Cubana de Noticias', 'Al Hashmi', 'Andrea Margutti Trophy - OKJ[clarification', 'Andrea de Cesaris.[53][54', 'Antonio Fuoco.[27][28', 'Appendix 1 - ^ "', 'Arrow McLaren Rounds', 'Arrow Schmidt Peterson', 'Arrow Schmidt Peterson Motorsports', 'Aston Martin F1.com', 'Atlas F1', 'Audi Sport ABT Schaeffler', 'Austria 2023 - Championship', 'Autodrom Igora Drive', 'Avio Costruzioni Ferrari', 'Axalta Racing', 'Ayrton Senna Alain', 'Azerbaijan Grands', 'BBC Panorama']


In [25]:
found_df = linked_df[linked_df["wikidata_uri"].notna()]

print("Trouvés par type:")
print(found_df["entity_type"].value_counts())

print("\nExemples trouvés (Driver, confidence=1.0):")
print(found_df[
    (found_df["entity_type"] == "Driver") & 
    (found_df["confidence"] == 1.0)
]["entity_text"].head(20).tolist())

Trouvés par type:
entity_type
Driver       415
GrandPrix     50
Team          10
Name: count, dtype: int64

Exemples trouvés (Driver, confidence=1.0):
['Achille Varzi', 'Adrian Newey', 'Alan Jones', 'Alberto Ascari', 'Alessandro Pier Guidi', 'Alex Lloyd', 'Alexander Albon', 'Alexander Rossi', 'Alfa Romeo', 'Andrea Kimi Antonelli', 'Andrea de Adamich', 'Anthony Kumpen', 'Antonio Fuoco', 'Antonio Giovinazzi', 'Ayrton Senna', 'Brendon Leigh', 'Brian Barnhart', 'Bruno Giacomelli', 'Bruno Senna', 'Carlos Reutemann']


In [26]:
from src.kg.alignment_builder import build_alignment_graph, get_aligned_qids

alignment_graph = build_alignment_graph(linked)

print(f"Triples d'alignement: {len(alignment_graph)}")

qids = get_aligned_qids(linked, min_confidence=0.6)
print(f"QIDs pour expansion: {len(qids)}")

Triples d'alignement: 949
QIDs pour expansion: 475


In [27]:
alignment_graph.serialize("../kg_artifacts/alignment.ttl", format="turtle")
print("Sauvegardé → kg_artifacts/alignment.ttl")

Sauvegardé → kg_artifacts/alignment.ttl


In [29]:
import requests

url = "https://query.wikidata.org/sparql"
headers = {
    "User-Agent": "F1-KG-Project/1.0 (student project; paula@example.com)",
    "Accept": "application/sparql-results+json",
}

query = """
SELECT ?p ?o WHERE {
  wd:Q9673 ?p ?o .
}
LIMIT 10
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=30)
print("Status:", response.status_code)
print("Response:", response.text[:500])

Status: 200
Response: {
  "head" : {
    "vars" : [ "p", "o" ]
  },
  "results" : {
    "bindings" : [ {
      "p" : {
        "type" : "uri",
        "value" : "http://schema.org/version"
      },
      "o" : {
        "datatype" : "http://www.w3.org/2001/XMLSchema#integer",
        "type" : "literal",
        "value" : "2474909640"
      }
    }, {
      "p" : {
        "type" : "uri",
        "value" : "http://schema.org/dateModified"
      },
      "o" : {
        "datatype" : "http://www.w3.org/2001/XMLSchema#da


In [32]:
print("Status:", response.status_code)
print("Response:", response.text[:200])

Status: 502
Response: <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.18.0</center>
</body>
</html>



In [33]:
import time
time.sleep(30)

query = """
SELECT ?p ?o WHERE {
  wd:Q9673 ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 20
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=30)
print("Status:", response.status_code)

if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    for row in bindings:
        pred = row["p"]["value"]
        obj  = row["o"]["value"]
        print(f"{pred.split('/')[-1]:10} | {obj[:60]}")

Status: 200
P18        | http://commons.wikimedia.org/wiki/Special:FilePath/Prime%20M
P19        | http://www.wikidata.org/entity/Q19795
P21        | http://www.wikidata.org/entity/Q6581097
P22        | http://www.wikidata.org/entity/Q134614524
P27        | http://www.wikidata.org/entity/Q145
P31        | http://www.wikidata.org/entity/Q5
P54        | http://www.wikidata.org/entity/Q169898
P54        | http://www.wikidata.org/entity/Q172030
P54        | http://www.wikidata.org/entity/Q172721
P69        | http://www.wikidata.org/entity/Q7743376
P97        | http://www.wikidata.org/entity/Q102083
P97        | http://www.wikidata.org/entity/Q11105296
P106       | http://www.wikidata.org/entity/Q10349745
P106       | http://www.wikidata.org/entity/Q10841764
P109       | http://commons.wikimedia.org/wiki/Special:FilePath/Signature
P140       | http://www.wikidata.org/entity/Q9592
P166       | http://www.wikidata.org/entity/Q680221
P166       | http://www.wikidata.org/entity/Q1061233
P166   

In [35]:
from src.kg.sparql_expander import expand_one_hop

for qid in ["Q9673", "Q169898", "Q193662"]:
    raw = expand_one_hop(qid, limit=50)
    print(f"{qid} → {len(raw)} triples")

Q9673 → 8 triples
Q169898 → 14 triples
Q193662 → 4 triples


In [36]:
from src.kg.sparql_expander import expand_all

# saisons présentes dans le KB
season_years = entity_df[entity_df["entity_type"] == "Season"]["entity_text"].unique().tolist()
print(f"Saisons: {season_years}")

expansion_graph = expand_all(
    qids=qids,
    season_years=season_years,
    one_hop_limit=200,
    season_limit=1000,
    delay=1.0,
)

print(f"\nTotal triples expansion: {len(expansion_graph)}")

Saisons: ['1351', '1892', '1929', '1931', '1932', '1933', '1934', '1935', '1937', '1938', '1939', '1940', '1943', '1944', '1945', '1947', '1948', '1949', '1950', '1951', '1952', '1953', '1954', '1955', '1956', '1957', '1958', '1959', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026', '2027', '2030']
[sparql_expander] Expansion 1-hop pour 475 entités...
  50/475 entités traitées — 310 triples
  100/475 entités traitées — 655 triples
  150/475 entités traitées — 950 triples
  200/475 entités traitées — 1262 trip

In [37]:
# filtre les saisons F1 valides (1950 à aujourd'hui)
season_years = [
    y for y in entity_df[entity_df["entity_type"] == "Season"]["entity_text"].unique().tolist()
    if y.isdigit() and 1950 <= int(y) <= 2026
]
print(f"Saisons F1 valides: {season_years}")

Saisons F1 valides: ['1950', '1951', '1952', '1953', '1954', '1955', '1956', '1957', '1958', '1959', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']


In [38]:
import requests, time

url = "https://query.wikidata.org/sparql"
headers = {
    "User-Agent": "F1-KG-Project/1.0 (student project; paula@example.com)",
    "Accept": "application/sparql-results+json",
}

query = """
SELECT ?race ?p ?o WHERE {
  ?race wdt:P31 wd:Q941966 .
  ?race wdt:P585 ?date .
  FILTER(YEAR(?date) = 2024)
  ?race ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 50
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=30)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")
    for row in bindings[:5]:
        print(row["race"]["value"].split("/")[-1], "|", row["p"]["value"].split("/")[-1], "|", row["o"]["value"][:50])

Status: 200
Résultats: 0


In [39]:
# cherchons les prédicats du Grand Prix de Monaco 2024 (Q123482134)
query = """
SELECT ?p ?o WHERE {
  wd:Q123482134 ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 30
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=30)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    for row in bindings:
        print(row["p"]["value"].split("/")[-1], "|", row["o"]["value"][:60])

Status: 200
P31 | http://www.wikidata.org/entity/Q3305213
P136 | http://www.wikidata.org/entity/Q1047337
P170 | http://www.wikidata.org/entity/Q2048944
P186 | http://www.wikidata.org/entity/Q296955
P186 | http://www.wikidata.org/entity/Q12321255
P350 | 61917
P571 | 1890-01-01T00:00:00Z
P1476 | Twee mannen bekijken een schilderij aan de wand
P2048 | 46.5
P2049 | 73.5


In [43]:
import time
time.sleep(30)

# Qiji F1 Grand Prix = Q1968561 (Formula One race)
query_search = """
SELECT ?race ?label WHERE {
  ?race wdt:P31 wd:Q1968561 .
  ?race rdfs:label ?label .
  FILTER(LANG(?label) = "en")
  FILTER(CONTAINS(?label, "2024"))
}
LIMIT 10
"""

response = requests.get(url, params={"query": query_search, "format": "json"}, headers=headers, timeout=60)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")
    for row in bindings[:5]:
        print(row["race"]["value"].split("/")[-1], "|", row["label"]["value"])

Status: 200
Résultats: 0


In [44]:
time.sleep(10)

query_search = """
SELECT ?race ?type ?label WHERE {
  ?race rdfs:label "2024 Monaco Grand Prix"@en .
  ?race wdt:P31 ?type .
  ?race rdfs:label ?label .
  FILTER(LANG(?label) = "en")
}
LIMIT 5
"""

response = requests.get(url, params={"query": query_search, "format": "json"}, headers=headers, timeout=60)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")
    for row in bindings:
        print(row["race"]["value"].split("/")[-1], "|", row["type"]["value"].split("/")[-1], "|", row["label"]["value"])

Status: 200
Résultats: 2
Q115910301 | Q9102 | 2024 Monaco Grand Prix
Q115910301 | Q11924610 | 2024 Monaco Grand Prix


In [46]:
time.sleep(10)

# on regarde tous les prédicats de Q115910301 (2024 Monaco Grand Prix)
query = """
SELECT ?p ?o WHERE {
  wd:Q115910301 ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 30
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=60)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")
    for row in bindings:
        print(row["p"]["value"].split("/")[-1], "|", row["o"]["value"][:60])

Status: 200
Résultats: 17
P17 | http://www.wikidata.org/entity/Q235
P18 | http://commons.wikimedia.org/wiki/Special:FilePath/Circuit%2
P31 | http://www.wikidata.org/entity/Q9102
P31 | http://www.wikidata.org/entity/Q11924610
P276 | http://www.wikidata.org/entity/Q45240
P276 | http://www.wikidata.org/entity/Q171400
P361 | http://www.wikidata.org/entity/Q113628886
P580 | 2024-05-24T00:00:00Z
P582 | 2024-05-26T00:00:00Z
P641 | http://www.wikidata.org/entity/Q1968
P641 | http://www.wikidata.org/entity/Q2705092
P1346 | http://www.wikidata.org/entity/Q169898
P1346 | http://www.wikidata.org/entity/Q17541912
P1705 | Formula 1 Grand Prix de Monaco 2024
P2671 | /g/11l7996309
P3157 | 78
P3450 | http://www.wikidata.org/entity/Q9102
